<table width="100%" bgcolor="#f4f5f6" style="width: 100%; background-color: #f4f5f6; border-left: 6px solid #ff9900; border-radius: 8px; border-collapse: collapse;">
  <tr>
    <td width="110" align="center" valign="middle" style="padding: 15px;">
      <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block;">
    </td>
    <td valign="middle" style="padding: 15px 20px 15px 5px; font-family: Arial, sans-serif;">
      <p style="margin: 0 0 4px 0; font-size: 13px; color: #555555;">Excerpt from the Amazon book</p>
      <h2 style="margin: 4px 0 10px 0; font-size: 20px; font-weight: 900; color: #111111; border: none; padding: 0;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</h2>
      <p style="margin: 0; font-size: 14px; font-weight: bold;">
        <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: underline;">🛒 Buy on Amazon &rarr;</a>
      </p>
    </td>
  </tr>
</table>

In [4]:
# 1. Install libraries
!pip install -q --no-warn-conflicts pyspark==3.5.3 delta-spark==3.2.1

# 2. Clone repository (sales.csv must be in Lakehouse/Files/)
!rm -rf shining-path_DP700
!git clone -q https://github.com/sansalamanca/shining-path_DP700.git

# 3. Initiate SparkSession
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Delta_Upsert_Demo") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print("¡SparkSession y repositorio listos!")

¡SparkSession y repositorio listos!


In [5]:
from delta.tables import DeltaTable

# Path for our target Delta table inside Tables/
target_table_path = "shining-path_DP700/Lakehouse/Tables/fact_sales"

# 1. Load initial data and save it as our existing target Delta table (e.g., first 10 rows)
df_initial = spark.read.format("csv") \
    .option("header", "true") \
    .option("escape", '"') \
    .load("shining-path_DP700/Lakehouse/Files/sales.csv") \
    .limit(10)

df_initial.write.format("delta").mode("overwrite").save(target_table_path)
print("Initial target Delta table created with 10 rows.")

Initial target Delta table created with 10 rows.


In [6]:
# 2. Simulate incoming daily updates (Source DataFrame)
# We load the first 15 rows from the source CSV to create a mix of existing records (to update) and new records (to insert)
df_updates = spark.read.format("csv") \
    .option("header", "true") \
    .option("escape", '"') \
    .load("shining-path_DP700/Lakehouse/Files/sales.csv") \
    .limit(15)

print(f"Source updates DataFrame ready with {df_updates.count()} rows.")

Source updates DataFrame ready with 15 rows.


In [7]:
# 3. Reference the target Delta table by path
target_delta_table = DeltaTable.forPath(spark, target_table_path)

# 4. Execute the MERGE (Upsert) matching by SalesOrderNumber and SalesOrderLineNumber
target_delta_table.alias("target").merge(
    source = df_updates.alias("source"),
    condition = "target.SalesOrderNumber = source.SalesOrderNumber AND target.SalesOrderLineNumber = source.SalesOrderLineNumber"
).whenMatchedUpdate(set={
    "Quantity": "source.Quantity",
    "UnitPrice": "source.UnitPrice",
    "TaxAmount": "source.TaxAmount"
}).whenNotMatchedInsert(values={
    "SalesOrderNumber": "source.SalesOrderNumber",
    "SalesOrderLineNumber": "source.SalesOrderLineNumber",
    "OrderDate": "source.OrderDate",
    "CustomerName": "source.CustomerName",
    "EmailAddress": "source.EmailAddress",
    "Item": "source.Item",
    "Quantity": "source.Quantity",
    "UnitPrice": "source.UnitPrice",
    "TaxAmount": "source.TaxAmount"
}).execute()

print("MERGE (UPSERT) operation executed successfully!")

MERGE (UPSERT) operation executed successfully!


In [8]:
# 5. Read back the target table to verify updates and new inserts
df_result = spark.read.format("delta").load(target_table_path)

print(f"Total rows in target table after UPSERT: {df_result.count()}")
df_result.select("SalesOrderNumber", "CustomerName", "Item", "Quantity", "UnitPrice").show(15, truncate=False)

Total rows in target table after UPSERT: 15
+----------------+----------------+-----------------------+--------+---------+
|SalesOrderNumber|CustomerName    |Item                   |Quantity|UnitPrice|
+----------------+----------------+-----------------------+--------+---------+
|SO43697         |Cole Watson     |Road-150 Red, 62       |1       |3578.27  |
|SO43698         |Rachael Martinez|Mountain-100 Silver, 44|1       |3399.99  |
|SO43699         |Sydney Wright   |Mountain-100 Silver, 44|1       |3399.99  |
|SO43700         |Ruben Prasad    |Road-650 Black, 62     |1       |699.0982 |
|SO43701         |Christy Zhu     |Mountain-100 Silver, 44|1       |3399.99  |
|SO43702         |Colin Anand     |Road-150 Red, 44       |1       |3578.27  |
|SO43703         |Albert Alvarez  |Road-150 Red, 62       |1       |3578.27  |
|SO43704         |Julio Ruiz      |Mountain-100 Black, 48 |1       |3374.99  |
|SO43705         |Curtis Lu       |Mountain-100 Silver, 38|1       |3399.99  |
|SO43706